In [1]:
import numpy as np
import torch
import torch.nn as nn

In [58]:
# Parameters (feel free to change for testing)
C = 3              # in_channels
out_channels = C   # out_channels
H, W = 5, 5        # spatial dims
kernelSize = 4
stride = 2         # can be >1 if needed
padding = 1        # can adjust
tfStruct = False   # toggle this to match the ND4J conditional path

# 1️⃣ Create input with input[0, c, h, w] = c + h + w
#input_np = (np.random.uniform(0, 10, (1, C, H, W))).astype(np.float32)
input_np = np.zeros((1, C, H, W), dtype=np.float32)
for c in range(C):
    for h in range(H):
        for w in range(W):
            input_np[0, c, h, w] = c + h + w

input_tensor = torch.from_numpy(input_np)

# 2️⃣ Create kernel weights
if tfStruct:
    # Shape: (kH, kW, C, outC) → needs permuting for PyTorch
    kernel_np = np.zeros((kernelSize, kernelSize, C, out_channels), dtype=np.float32)
    for f in range(out_channels):
        for c in range(C):
            kernel_np[:, :, c, f] = (f + 1) * (c + 1)
    # Convert to PyTorch layout: (outC, inC, kH, kW)
    kernel_pt = np.transpose(kernel_np, (3, 2, 0, 1))
else:
    # Shape: (outC, C, kH, kW)
    kernel_pt = np.zeros((out_channels, C, kernelSize, kernelSize), dtype=np.float32)
    for f in range(out_channels):
        for c in range(C):
            kernel_pt[f, c, :, :] = (f + 1) * (c + 1)

#kernel_pt = (np.random.uniform(-3, 3, (out_channels, C, kernelSize, kernelSize))).astype(np.float32)

In [59]:
# 3️⃣ Create ConvTranspose2d layer and manually load weights
convT = nn.ConvTranspose2d(in_channels=C,
                           out_channels=out_channels,
                           kernel_size=kernelSize,
                           stride=stride,
                           padding=padding,
                           bias=False)

In [60]:
# Replace weights
with torch.no_grad():
    convT.weight.copy_(torch.tensor(kernel_pt))

# 4️⃣ Apply transpose convolution
output = convT(input_tensor)

# ✅ Done
print("Input shape:", input_tensor.shape)
print("Kernel shape:", convT.weight.shape)
print("Output shape:", output.shape)
print("Input tensor:\n", input_tensor)
print("Weight tensor:\n", kernel_pt)
print("Output tensor:\n", output)

Input shape: torch.Size([1, 3, 5, 5])
Kernel shape: torch.Size([3, 3, 4, 4])
Output shape: torch.Size([1, 3, 10, 10])
Input tensor:
 tensor([[[[ 0.,  1.,  2.,  3.,  4.],
          [ 1.,  2.,  3.,  4.,  5.],
          [ 2.,  3.,  4.,  5.,  6.],
          [ 3.,  4.,  5.,  6.,  7.],
          [ 4.,  5.,  6.,  7.,  8.]],

         [[ 1.,  2.,  3.,  4.,  5.],
          [ 2.,  3.,  4.,  5.,  6.],
          [ 3.,  4.,  5.,  6.,  7.],
          [ 4.,  5.,  6.,  7.,  8.],
          [ 5.,  6.,  7.,  8.,  9.]],

         [[ 2.,  3.,  4.,  5.,  6.],
          [ 3.,  4.,  5.,  6.,  7.],
          [ 4.,  5.,  6.,  7.,  8.],
          [ 5.,  6.,  7.,  8.,  9.],
          [ 6.,  7.,  8.,  9., 10.]]]])
Weight tensor:
 [[[[1. 1. 1. 1.]
   [1. 1. 1. 1.]
   [1. 1. 1. 1.]
   [1. 1. 1. 1.]]

  [[2. 2. 2. 2.]
   [2. 2. 2. 2.]
   [2. 2. 2. 2.]
   [2. 2. 2. 2.]]

  [[3. 3. 3. 3.]
   [3. 3. 3. 3.]
   [3. 3. 3. 3.]
   [3. 3. 3. 3.]]]


 [[[2. 2. 2. 2.]
   [2. 2. 2. 2.]
   [2. 2. 2. 2.]
   [2. 2. 2. 2.]]

  [[4. 

In [49]:
np.save("Conv16x16s2p1ic128oc256-rand-out.npy", output.detach().numpy())

In [50]:
np.save("Conv16x16s2p1ic128oc256-rand-in.npy", input_np)
np.save("Conv16x16s2p1ic128oc256-rand-w.npy", kernel_pt)

In [44]:
# Input array: shape (1, 3, 5, 5)
input_array = np.array([[
    # Channel 0
    [
        [1, 2, 3, 4, 5],
        [6, 7, 8, 9, 10],
        [11, 12, 13, 14, 15],
        [16, 17, 18, 19, 20],
        [21, 22, 23, 24, 25]
    ],
    # Channel 1
    [
        [0.1, 0.2, 0.3, 0.4, 0.5],
        [0.6, 0.7, 0.8, 0.9, 1.0],
        [1.1, 1.2, 1.3, 1.4, 1.5],
        [1.6, 1.7, 1.8, 1.9, 2.0],
        [2.1, 2.2, 2.3, 2.4, 2.5]
    ],
    # Channel 2
    [
        [-1, -2, -3, -4, -5],
        [-6, -7, -8, -9, -10],
        [-11, -12, -13, -14, -15],
        [-16, -17, -18, -19, -20],
        [-21, -22, -23, -24, -25]
    ]
]])

input_tensor = torch.from_numpy(input_array.astype(np.float32))

# Kernel array: shape (4, 3, 2, 2)
kernel = np.array([
    # Output channel 0
    [
        [[1, 0], [0, 1]],  # Input channel 0
        [[0, 1], [1, 0]],  # Input channel 1
        [[1, 1], [1, 1]]   # Input channel 2
    ],
]).astype(np.float32)


In [45]:
kernel.reshape(3,1,2,2)

array([[[[1., 0.],
         [0., 1.]]],


       [[[0., 1.],
         [1., 0.]]],


       [[[1., 1.],
         [1., 1.]]]], dtype=float32)

In [46]:
# 3️⃣ Create ConvTranspose2d layer and manually load weights
convT = nn.ConvTranspose2d(in_channels=3,
                           out_channels=1,
                           kernel_size=2,
                           stride=2,
                           padding=0,
                           bias=False)

In [47]:
# Replace weights
with torch.no_grad():
    convT.weight.copy_(torch.tensor(kernel.reshape(3,1,2,2)))

# 4️⃣ Apply transpose convolution
output = convT(input_tensor)

# ✅ Done
print("Input shape:", input_tensor.shape)
print("Kernel shape:", convT.weight.shape)
print("Output shape:", output.shape)
print("Input tensor:\n", input_tensor)
print("Weight tensor:\n", kernel.reshape(3,1,2,2))
print("Output tensor:\n", output)

Input shape: torch.Size([1, 3, 5, 5])
Kernel shape: torch.Size([3, 1, 2, 2])
Output shape: torch.Size([1, 1, 10, 10])
Input tensor:
 tensor([[[[  1.0000,   2.0000,   3.0000,   4.0000,   5.0000],
          [  6.0000,   7.0000,   8.0000,   9.0000,  10.0000],
          [ 11.0000,  12.0000,  13.0000,  14.0000,  15.0000],
          [ 16.0000,  17.0000,  18.0000,  19.0000,  20.0000],
          [ 21.0000,  22.0000,  23.0000,  24.0000,  25.0000]],

         [[  0.1000,   0.2000,   0.3000,   0.4000,   0.5000],
          [  0.6000,   0.7000,   0.8000,   0.9000,   1.0000],
          [  1.1000,   1.2000,   1.3000,   1.4000,   1.5000],
          [  1.6000,   1.7000,   1.8000,   1.9000,   2.0000],
          [  2.1000,   2.2000,   2.3000,   2.4000,   2.5000]],

         [[ -1.0000,  -2.0000,  -3.0000,  -4.0000,  -5.0000],
          [ -6.0000,  -7.0000,  -8.0000,  -9.0000, -10.0000],
          [-11.0000, -12.0000, -13.0000, -14.0000, -15.0000],
          [-16.0000, -17.0000, -18.0000, -19.0000, -20.00

In [ ]:
np.save("Transp-test-out.npy", output.detach().numpy())
np.save("Transp-test-in.npy", input_tensor)
np.save("Transp-test-w.npy", kernel)

In [ ]:
import torch
import torch.nn as nn

convM = nn.ConvTranspose2d(in_channels=3, out_channels=1, kernel_size=2, stride=2, bias=False)
print("Weight shape:", convM.weight.shape) #I literally cannot believe this

Weight shape: torch.Size([3, 1, 2, 2])


In [5]:
ker = (np.load("../src/main/java/fusion/correct_outs/transp/TranspConv10x10s2p1c128-rand-w.npy")).transpose(1,0,2,3)
inp = torch.tensor(np.load("../src/main/java/fusion/correct_outs/transp/TranspConv10x10s2p1c128-rand-in.npy"))
convT = nn.ConvTranspose2d(in_channels=128,
                           out_channels=128,
                           kernel_size=4,
                           stride=2,
                           padding=1,
                           bias=False)

# Replace weights
with torch.no_grad():
    convT.weight.copy_(torch.tensor(ker))

# 4️⃣ Apply transpose convolution
output = convT(inp)

# ✅ Done
print("Input shape:", inp.shape)
print("Kernel shape:", convT.weight.shape)
print("Output shape:", output.shape)
print("Input tensor:\n", inp)
print("Weight tensor:\n", ker.transpose(1,0,2,3))
print("Output tensor:\n", output)

Input shape: torch.Size([1, 128, 10, 10])
Kernel shape: torch.Size([128, 128, 4, 4])
Output shape: torch.Size([1, 128, 20, 20])
Input tensor:
 tensor([[[[8.6039, 6.5928, 4.9984,  ..., 4.2486, 9.7725, 2.1017],
          [4.2703, 3.5773, 7.0587,  ..., 3.2986, 4.7648, 4.9172],
          [3.8886, 6.8673, 2.6382,  ..., 3.8105, 2.5363, 0.1926],
          ...,
          [6.6377, 9.5166, 9.8106,  ..., 9.1886, 7.6130, 2.6336],
          [6.6640, 4.1711, 3.9282,  ..., 8.9801, 5.8307, 8.7760],
          [0.2809, 9.4492, 4.2686,  ..., 5.6327, 3.5309, 3.6372]],

         [[3.8802, 0.4413, 5.0599,  ..., 2.8044, 5.3444, 2.5424],
          [9.0144, 7.7836, 1.3250,  ..., 6.3891, 9.7191, 3.2250],
          [9.3292, 5.0272, 5.0341,  ..., 7.9748, 9.0065, 5.6822],
          ...,
          [3.3611, 6.1398, 2.2948,  ..., 3.8487, 7.0563, 3.4360],
          [2.4240, 7.4314, 7.5432,  ..., 7.9172, 4.1901, 5.8040],
          [3.3829, 5.4509, 8.5787,  ..., 1.1712, 4.6707, 0.7861]],

         [[1.4277, 5.1404, 7.98

In [6]:
np.save("TranspConv10x10s2p1c128-rand-out.npy", output.detach().numpy())